# PhishGuard AI — Intelligent Phishing Email Detection
### End-to-End Machine Learning & Threat Engineering Pipeline
---
This notebook demonstrates the complete data science and ML engineering workflow:
1. **Exploratory Data Analysis (EDA):** Ingestion of 18,000+ authentic email records.
2. **Text Normalization & Sanitization:** HTML tag stripping, whitespace standardization, and casing.
3. **Hybrid Feature Engineering:** Sublinear TF-IDF (unigrams + bigrams) combined with structural URL and keyword heuristics.
4. **Model Comparison:** Logistic Regression versus Multinomial Naive Bayes.
5. **Evaluation:** Precision, Recall, F1-Score, and Confusion Matrix.
6. **Inference Demo:** Realistic threat evaluation.

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Ensure project modules are importable
sys.path.insert(0, os.path.abspath('..'))
import features

print('Core libraries and features.py imported successfully.')

## 1. Load Dataset & Inspect Distribution

In [2]:
csv_path = os.path.join('..', 'data', 'Phishing_Email.csv')
df = pd.read_csv(csv_path)
print('Dataset Shape:', df.shape)
print('\nClass Counts:')
print(df['Email Type'].value_counts())

## 2. Text Preprocessing & Cleaning
Drop nulls, clean HTML elements, and encode labels.

In [3]:
df = df.dropna(subset=['Email Text', 'Email Type']).copy()
df['target'] = df['Email Type'].apply(lambda x: 1 if 'phish' in str(x).lower() else 0)
df['cleaned_text'] = df['Email Text'].apply(features.clean_email_text)
df = df[df['cleaned_text'].str.len() > 5].copy()
print(f'Valid cleaned samples: {len(df)}')
df[['Email Text', 'cleaned_text', 'target']].head(2)

## 3. Stratified Train/Test Split & Feature Extraction

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'].tolist(), df['target'].to_numpy(),
    test_size=0.20, random_state=42, stratify=df['target'].to_numpy()
)
print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')

## 4. Evaluated Results & Champion Model Metrics
Read the generated metrics from the training pipeline.

In [5]:
metrics_file = os.path.join('..', 'model', 'metrics.json')
with open(metrics_file, 'r', encoding='utf-8') as f:
    metrics = json.load(f)

print('Selected Champion Model:', metrics['model_name'])
print(f'Accuracy   : {metrics["accuracy"]}%')
print(f'Precision  : {metrics["precision"]}%')
print(f'Recall     : {metrics["recall"]}%')
print(f'F1-Score   : {metrics["f1_score"]}%')
print('\nModel Comparison:')
for m, scores in metrics['comparison'].items():
    print(f'  {m}: Acc={scores["accuracy"]}% | F1={scores["f1_score"]}%')

## 5. Live Inference Demonstration
Test with realistic phishing and legitimate email payloads.

In [6]:
from app import predict_email

phishing_text = 'URGENT: Your account was suspended! Verify password at http://192.168.1.1/login or bit.ly/auth immediately.'
safe_text = 'Hi team, please find the quarterly report attached. See you at tomorrow sync meeting.'

print('--- Test 1 (Phishing) ---')
print(predict_email(phishing_text))
print('\n--- Test 2 (Safe) ---')
print(predict_email(safe_text))